# GCP Streaming and Batch — Pub/Sub, Dataflow, Dataproc

## Mental Model

This notebook shows the **canonical GCP streaming path** for enterprise telemetry:

**Producers → Pub/Sub → Dataflow → BigQuery**

For Citi-style observability workloads, Pub/Sub handles bursty event ingestion and fan-out, Dataflow handles stream or batch transforms, and Dataproc is the managed Spark/Hadoop option when you need a cluster-shaped runtime instead of Beam.

We will work with this fixed domain context:

- **Project**: `citi-de-learning`
- **PostgreSQL**: `localhost:5432`, database `de_telemetry`
- **Kafka**: `localhost:9092`
- **Spark**: `pyspark==3.5.4`
- **Airflow**: `localhost:8082`
- **MLflow**: `localhost:5000`
- **dbt project**: `citi_dbt`
- **Databricks warehouse**: `b6657f31d1e7a179`

### Citi narrative

Think of the source data as telemetry for **6,000+ API endpoints** where latency, error rate, throughput, and operational alerting must be routed to multiple consumers at once.

### Dataset context

- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows


In [ ]:
from __future__ import annotations

import json
import os
import time
import uuid
from datetime import datetime, timezone
from pprint import pprint

PROJECT_ID = "citi-de-learning"
GOOGLE_APPLICATION_CREDENTIALS = r"D:/Workspace/Technologies/_setup/gcp_key.json"

# Use the real path from the provided environment.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GOOGLE_APPLICATION_CREDENTIALS

print("PROJECT_ID:", PROJECT_ID)
print("GOOGLE_APPLICATION_CREDENTIALS:", os.environ["GOOGLE_APPLICATION_CREDENTIALS"])
print("Credential file exists:", os.path.exists(os.environ["GOOGLE_APPLICATION_CREDENTIALS"]))


## Setup

Packages are assumed to be pre-installed. No `%pip install` cells are used.

This notebook uses:

- `google-cloud-pubsub`
- `google-auth`
- `requests`

The Pub/Sub section performs real work.
The Dataflow and Dataproc sections build production-grade request payloads and optionally execute API calls only when explicitly enabled, so the notebook remains safe to run end-to-end.


In [ ]:
from google.api_core.exceptions import AlreadyExists, NotFound
from google.cloud import pubsub_v1
import google.auth
from google.auth.transport.requests import Request
import requests

credentials, detected_project = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
credentials.refresh(Request())

print("Authenticated project from credentials:", detected_project)
print("Requested project:", PROJECT_ID)
print("Credentials valid:", credentials.valid)


## 1) Pub/Sub Advanced — Fan-out Pattern

We will create:

- Topic: `citi-telemetry-stream`
- Subscriptions:
  - `alerting-sub`
  - `analytics-sub`

Then we will:

1. Publish **50 alert messages**
2. Attach attributes such as `severity` and `region`
3. Pull from both subscriptions
4. Confirm that **both subscriptions receive all 50 messages**
5. Clean up topic and subscriptions

This demonstrates the Pub/Sub fan-out model: one published message can be independently consumed by multiple downstream subscribers.


In [ ]:
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

topic_id = "citi-telemetry-stream"
subscription_ids = ["alerting-sub", "analytics-sub"]

topic_path = publisher.topic_path(PROJECT_ID, topic_id)
subscription_paths = {
    sub_id: subscriber.subscription_path(PROJECT_ID, sub_id)
    for sub_id in subscription_ids
}

def ensure_topic(topic_path: str):
    try:
        publisher.create_topic(request={"name": topic_path})
        print(f"Created topic: {topic_path}")
    except AlreadyExists:
        print(f"Topic already exists: {topic_path}")

def ensure_subscription(sub_path: str, topic_path: str):
    try:
        subscriber.create_subscription(
            request={
                "name": sub_path,
                "topic": topic_path,
                "ack_deadline_seconds": 30,
            }
        )
        print(f"Created subscription: {sub_path}")
    except AlreadyExists:
        print(f"Subscription already exists: {sub_path}")

ensure_topic(topic_path)
for sub_id, sub_path in subscription_paths.items():
    ensure_subscription(sub_path, topic_path)


In [ ]:
severities = ["critical", "high", "medium", "low"]
regions = ["us-east1", "us-central1", "europe-west1", "asia-south1"]

publish_futures = []
published_payloads = []

for i in range(50):
    payload = {
        "alert_id": i + 1,
        "endpoint_id": 1000 + i,
        "message": f"Telemetry alert {i+1} for Citi API endpoint",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "event_type": "telemetry_alert",
        "run_id": str(uuid.uuid4()),
    }
    severity = severities[i % len(severities)]
    region = regions[i % len(regions)]

    future = publisher.publish(
        topic_path,
        json.dumps(payload).encode("utf-8"),
        severity=severity,
        region=region,
        source="de_telemetry",
        dataset="alerts",
    )
    publish_futures.append(future)
    published_payloads.append((payload, severity, region))

message_ids = [future.result(timeout=30) for future in publish_futures]
print("Published messages:", len(message_ids))
print("First 3 message IDs:", message_ids[:3])


In [ ]:
def pull_exactly(subscription_path: str, expected_count: int, max_rounds: int = 20, batch_size: int = 10):
    received = []
    seen_message_ids = set()

    for round_no in range(1, max_rounds + 1):
        response = subscriber.pull(
            request={
                "subscription": subscription_path,
                "max_messages": batch_size,
            },
            timeout=15,
        )

        if not response.received_messages:
            print(f"[{subscription_path}] round {round_no}: no messages returned")
            time.sleep(1)
            continue

        ack_ids = []
        for rm in response.received_messages:
            ack_ids.append(rm.ack_id)
            if rm.message.message_id not in seen_message_ids:
                seen_message_ids.add(rm.message.message_id)
                received.append({
                    "message_id": rm.message.message_id,
                    "data": json.loads(rm.message.data.decode("utf-8")),
                    "attributes": dict(rm.message.attributes),
                    "publish_time": rm.message.publish_time.isoformat(),
                })

        subscriber.acknowledge(
            request={
                "subscription": subscription_path,
                "ack_ids": ack_ids,
            }
        )

        print(f"[{subscription_path}] round {round_no}: total unique received = {len(received)}")

        if len(received) >= expected_count:
            break

        time.sleep(1)

    return received

received_by_subscription = {}
for sub_id, sub_path in subscription_paths.items():
    received_by_subscription[sub_id] = pull_exactly(sub_path, expected_count=50)

summary = {sub_id: len(msgs) for sub_id, msgs in received_by_subscription.items()}
print("Received counts by subscription:")
pprint(summary)

assert summary["alerting-sub"] == 50, f"alerting-sub received {summary['alerting-sub']} instead of 50"
assert summary["analytics-sub"] == 50, f"analytics-sub received {summary['analytics-sub']} instead of 50"

print("Fan-out verification passed: both subscriptions received all 50 messages.")


In [ ]:
sample_alerting = received_by_subscription["alerting-sub"][0]
sample_analytics = received_by_subscription["analytics-sub"][0]

print("Sample from alerting-sub:")
pprint(sample_alerting)

print("\nSample from analytics-sub:")
pprint(sample_analytics)


In [ ]:
# Cleanup in child-first order: subscriptions, then topic.
for sub_id, sub_path in subscription_paths.items():
    try:
        subscriber.delete_subscription(request={"subscription": sub_path})
        print(f"Deleted subscription: {sub_path}")
    except NotFound:
        print(f"Subscription already absent: {sub_path}")

try:
    publisher.delete_topic(request={"topic": topic_path})
    print(f"Deleted topic: {topic_path}")
except NotFound:
    print(f"Topic already absent: {topic_path}")


## 2) Dataflow Template — Apache Beam Mental Model

### Beam concepts

- **PCollection**: the distributed dataset abstraction
- **PTransform**: a transformation such as map, filter, group, window, join
- **Runner**: the execution engine

In GCP, **Dataflow** is the managed Beam runner.

### When to use Dataflow vs Spark

Use **Dataflow** when:
- you want a managed Beam runtime
- you need unified batch and streaming semantics
- you want autoscaling and lower operational burden
- your pipeline fits Beam transforms naturally

Use **Spark / Dataproc** when:
- your team already thinks in Spark
- you need cluster-level control
- you have Spark-native libraries or existing jobs
- you want deeper control over executors, shuffle, caching, and runtime tuning

Below we construct a **production-style Dataflow template launch request** using the classic **GCS Text to BigQuery** template.

To keep this notebook safe for repeated runs, the API call is **disabled by default**.
Set `RUN_DATAFLOW_TEMPLATE = True` only when you intentionally want to launch a real job.


In [ ]:
DATAFLOW_REGION = "us-central1"
DATAFLOW_TEMPLATE_GCS_PATH = "gs://dataflow-templates-us-central1/latest/GCS_Text_to_BigQuery"
DATAFLOW_JOB_NAME = f"citi-gcs-to-bq-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
DATAFLOW_STAGING_LOCATION = "gs://citi-de-learning-dataflow/staging"
DATAFLOW_TEMP_LOCATION = "gs://citi-de-learning-dataflow/temp"

dataflow_launch_url = (
    f"https://dataflow.googleapis.com/v1b3/projects/{PROJECT_ID}/locations/{DATAFLOW_REGION}/templates:launch"
)

dataflow_payload = {
    "jobName": DATAFLOW_JOB_NAME,
    "gcsPath": DATAFLOW_TEMPLATE_GCS_PATH,
    "parameters": {
        "javascriptTextTransformGcsPath": "gs://citi-de-learning-dataflow/transforms/identity.js",
        "javascriptTextTransformFunctionName": "transform",
        "JSONPath": "gs://citi-de-learning-dataflow/schemas/alerts_schema.json",
        "inputFilePattern": "gs://citi-de-learning-dataflow/input/alerts/*.json",
        "outputTable": f"{PROJECT_ID}:telemetry.alerts_raw",
        "bigQueryLoadingTemporaryDirectory": "gs://citi-de-learning-dataflow/bq_temp",
    },
    "environment": {
        "tempLocation": DATAFLOW_TEMP_LOCATION,
        "serviceAccountEmail": "",
        "additionalUserLabels": {
            "workload": "citi-telemetry",
            "mode": "template-launch"
        }
    }
}

print("Dataflow launch URL:")
print(dataflow_launch_url)
print("\nDataflow payload:")
print(json.dumps(dataflow_payload, indent=2))


In [ ]:
RUN_DATAFLOW_TEMPLATE = False

if RUN_DATAFLOW_TEMPLATE:
    headers = {
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json",
    }
    response = requests.post(
        dataflow_launch_url,
        headers=headers,
        json=dataflow_payload,
        timeout=60,
    )
    print("HTTP status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)
else:
    print(
        "Dataflow template launch skipped intentionally. "
        "Set RUN_DATAFLOW_TEMPLATE = True to execute the REST call."
    )


## 3) Dataproc — Managed Spark/Hadoop on GCP

Think of Dataproc as the **managed cluster option** for Spark and Hadoop on GCP.

### Why teams use Dataproc

- managed Spark/Hadoop/YARN clusters
- fast cluster startup
- autoscaling policies
- optional component gateway for notebook and UI access
- cheaper transient clusters for job-oriented workloads

### Operational ideas

- **Primary workers**: stable core capacity
- **Secondary/preemptible workers**: lower-cost burst capacity for tolerant jobs
- **Autoscaling**: expand or shrink based on backlog and utilization
- **Component Gateway**: easier access to Spark History Server, YARN UI, and related UIs

Below we build the **cluster create payload** and the matching **REST endpoint**.
We do **not** actually create the cluster by default to avoid cost.


In [ ]:
DATAPROC_REGION = "us-central1"
DATAPROC_CLUSTER_NAME = f"citi-dataproc-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
DATAPROC_IMAGE_VERSION = "2.2-debian12"
DATAPROC_BUCKET = "citi-de-learning-dataproc"
NETWORK_URI = "projects/citi-de-learning/global/networks/default"

dataproc_create_url = (
    f"https://dataproc.googleapis.com/v1/projects/{PROJECT_ID}/regions/{DATAPROC_REGION}/clusters"
)

dataproc_payload = {
    "projectId": PROJECT_ID,
    "clusterName": DATAPROC_CLUSTER_NAME,
    "config": {
        "gceClusterConfig": {
            "zoneUri": "us-central1-a",
            "networkUri": NETWORK_URI,
            "serviceAccountScopes": [
                "https://www.googleapis.com/auth/cloud-platform"
            ],
        },
        "masterConfig": {
            "numInstances": 1,
            "machineTypeUri": "e2-standard-4",
            "diskConfig": {
                "bootDiskType": "pd-standard",
                "bootDiskSizeGb": 100
            }
        },
        "workerConfig": {
            "numInstances": 2,
            "machineTypeUri": "e2-standard-4",
            "diskConfig": {
                "bootDiskType": "pd-standard",
                "bootDiskSizeGb": 100
            }
        },
        "secondaryWorkerConfig": {
            "numInstances": 2,
            "isPreemptible": True,
            "machineTypeUri": "e2-standard-4",
            "diskConfig": {
                "bootDiskType": "pd-standard",
                "bootDiskSizeGb": 100
            }
        },
        "softwareConfig": {
            "imageVersion": DATAPROC_IMAGE_VERSION,
            "optionalComponents": ["JUPYTER", "ZOOKEEPER"],
            "properties": {
                "spark:spark.sql.adaptive.enabled": "true"
            }
        },
        "endpointConfig": {
            "enableHttpPortAccess": True
        },
        "autoscalingConfig": {
            "policyUri": f"projects/{PROJECT_ID}/locations/{DATAPROC_REGION}/autoscalingPolicies/citi-autoscaling-policy"
        },
        "initializationActions": [
            {
                "executableFile": "gs://goog-dataproc-initialization-actions-us-central1/python/pip-install.sh",
                "executionTimeout": "600s"
            }
        ],
        "tempBucket": DATAPROC_BUCKET
    },
    "labels": {
        "env": "learning",
        "workload": "telemetry",
        "platform": "dataproc"
    }
}

print("Dataproc create URL:")
print(dataproc_create_url)
print("\nDataproc payload:")
print(json.dumps(dataproc_payload, indent=2))


In [ ]:
RUN_DATAPROC_CREATE = False

if RUN_DATAPROC_CREATE:
    headers = {
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json",
    }
    response = requests.post(
        dataproc_create_url,
        headers=headers,
        json=dataproc_payload,
        timeout=60,
    )
    print("HTTP status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)
else:
    print(
        "Dataproc cluster creation skipped intentionally. "
        "Set RUN_DATAPROC_CREATE = True to execute the REST call."
    )


## 4) GCP vs AWS Streaming Comparison

| Capability | GCP | AWS | Practical takeaway |
|---|---|---|---|
| Messaging / ingestion | Pub/Sub | Kinesis Data Streams / SNS+SQS patterns | Pub/Sub is the native GCP managed event bus |
| Stream processing | Dataflow | Kinesis Data Analytics / EMR / Glue Streaming | Dataflow is Beam-native and highly managed |
| Batch/cluster compute | Dataproc | EMR / Glue | Dataproc maps most directly to managed Spark/Hadoop |
| Serverless warehouse | BigQuery | Athena | BigQuery is warehouse-first; Athena is query-on-lake-first |
| Canonical streaming path | Pub/Sub → Dataflow → BigQuery | Kinesis → Lambda/Glue/EMR → S3/Redshift/Athena | Both are strong; operational shape differs |

### Intuition

- **Pub/Sub vs Kinesis**: both ingest streams, but Pub/Sub feels closer to a managed event backbone with simple fan-out semantics.
- **Dataflow vs EMR/Glue**: Dataflow is stronger when Beam and fully managed stream/batch are the center of gravity.
- **BigQuery vs Athena**: BigQuery is a more opinionated warehouse experience; Athena is lighter-weight over data already in S3.


## 5) What Just Happened

**Pub/Sub fan-out is GCP's answer to multiple consumer groups in Kafka.**  
A single topic fed two independent subscribers, and both received all 50 alerts.

**Dataflow is the operational Apache Beam runner.**  
You define the pipeline in Beam terms and let Dataflow own scaling and execution.

**Dataproc is the managed Spark/Hadoop lane.**  
Use it when your workload is cluster-shaped, Spark-native, or operationally closer to EMR-style thinking.

For Citi's GCP workloads, the canonical streaming pipeline is:

# Pub/Sub → Dataflow → BigQuery

That is the cloud-native answer for high-volume telemetry, alert enrichment, and downstream analytics.
